<a href="https://colab.research.google.com/github/GilliardMorandim/mba-tcc-usp-inadimplencia/blob/eda%2Ffeature/pre_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements

In [1]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark
!pip install mlxtend


In [2]:
import matplotlib.pyplot as plt
from google.colab import drive
import os
import pandas as pd

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from mlxtend.feature_selection import SequentialFeatureSelector as sfs
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

drive.mount('/content/drive')


Mounted at /content/drive


# Lendo Arquivo Pyspark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when
from pyspark.sql.functions import try_divide

spark = (SparkSession.builder
         .appName("LoadAllCSVs")
         .config("spark.driver.memory", "8g")
         .config("spark.executor.memory", "8g")
         .config("spark.sql.files.maxPartitionBytes", "256m")
         .getOrCreate())

print("Spark iniciado!")

Spark iniciado!


In [4]:
dfs_spark = {}

base_path = "/content/drive/MyDrive/MBA - Ciencia de Dados - USP/dados_tcc/"

files = [f for f in os.listdir(base_path) if f.endswith(".csv")]

for file in files:
    full_path = os.path.join(base_path, file)
    print(f"\n📥 Lendo via PySpark: {file}")

    try:
        df = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .csv(full_path))

        key = file.replace(".csv", "")
        dfs_spark[key] = df

        print(f"✔ OK - Linhas (estimado pela Spark): {df.count()} | Colunas: {len(df.columns)}")

    except Exception as e:
        print(f"❌ Erro ao ler {file}: {e}")

print("\nTodos os arquivos foram processados!")

globals().update(dfs_spark)


📥 Lendo via PySpark: pre_aprovado.csv
✔ OK - Linhas (estimado pela Spark): 42171363 | Colunas: 11

📥 Lendo via PySpark: parcelas.csv
✔ OK - Linhas (estimado pela Spark): 1865444 | Colunas: 21

📥 Lendo via PySpark: contratos.csv
✔ OK - Linhas (estimado pela Spark): 382539 | Colunas: 26

📥 Lendo via PySpark: score_credito.csv
✔ OK - Linhas (estimado pela Spark): 266126 | Colunas: 3

📥 Lendo via PySpark: analise_conversao.csv
✔ OK - Linhas (estimado pela Spark): 538023 | Colunas: 12

📥 Lendo via PySpark: analise_conversao_v2.csv
✔ OK - Linhas (estimado pela Spark): 538023 | Colunas: 12

📥 Lendo via PySpark: analise_conversao_v3.csv
✔ OK - Linhas (estimado pela Spark): 533525 | Colunas: 11

Todos os arquivos foram processados!


# Pre-Processing

In [20]:
parcelas_main = parcelas

contratos_keys = contratos.select(
    "id_contrato",
    "uuid_cliente",
    "id_contrato_original",
    "id_contrato_pai",
    "cpf_hash_sha256").filter("id_contrato IS NOT NULL")


parcelas_main = parcelas.join(
    contratos_keys,
    on="id_contrato",
    how="left"
)

parcelas_main = parcelas_main.filter("data_vencimento < '2025-11-01'")

# Normalização da parcela

In [21]:
# janela por contrato
w = Window.partitionBy("id_contrato")

parcelas_main = (
    parcelas_main

    .withColumn("min_parcela", F.min("parcela").over(w))
    .withColumn("max_parcela", F.max("parcela").over(w))
    .withColumn(
        "parcela_norm_0_1",
        F.round((F.col("parcela") - F.col("min_parcela")) /
        (F.col("max_parcela") - F.col("min_parcela")),2
    ))
    .drop("min_parcela", "max_parcela")
)

#parcelas_main.orderBy("data_vencimento").filter(F.col("id_contrato") == "{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}").show(100, truncate=False)
#parcelas_main.show(10,truncate=False)

In [22]:
#define a janela de particionamento de underbound
w_ffill =(Window.partitionBy("id_contrato")
          .orderBy("parcela")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow))

parcelas_main = parcelas_main.withColumn("data_pagamento_aux",F.last("data_pagamento",ignorenulls=True).over(w_ffill))

parcelas_main = parcelas_main.withColumn("data_vencimento_aux",F.greatest
 (F.datediff(F.col("data_pagamento_aux"),F.col("data_vencimento")), F.lit(0)))
parcelas_main = parcelas_main.drop("data_vencimento_aux")

In [23]:
#parcelas_main.orderBy("data_vencimento").filter(F.col("id_contrato") == "{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}").show(100, truncate=False)

# Flag Contrato sem nenhum pagamento

In [24]:
w_contrato = Window.partitionBy("id_contrato")

parcelas_main = parcelas_main.withColumn(
    "flag_contrato_sem_pagamento",
    F.when(
        F.max(F.col("data_pagamento").isNotNull().cast("int")).over(w_contrato) == 0,
        F.lit(1)
    ).otherwise(F.lit(0))
)


In [25]:
#Validando flag-inadimplencia over todas parcelas
parcelas_main.select(
    "id_contrato",
    "parcela",
    "data_vencimento",
    "data_pagamento",
    "data_pagamento_aux",
    "flag_contrato_sem_pagamento",
).orderBy("data_vencimento").filter(F.col("id_contrato") == "{00012C6D-DD36-401A-8B26-F0C46C121114}").show(100, truncate=False)

+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+
|id_contrato                           |parcela|data_vencimento|data_pagamento|data_pagamento_aux|flag_contrato_sem_pagamento|
+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+
|{00012C6D-DD36-401A-8B26-F0C46C121114}|1      |2025-07-25     |NULL          |NULL              |1                          |
|{00012C6D-DD36-401A-8B26-F0C46C121114}|2      |2025-08-25     |NULL          |NULL              |1                          |
|{00012C6D-DD36-401A-8B26-F0C46C121114}|3      |2025-09-25     |NULL          |NULL              |1                          |
+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+



# Data Pagamento Aux

In [26]:
w_primeira = (
    Window
    .partitionBy("id_contrato")
    .orderBy("parcela")
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)

# Criação da coluna com a data de vencimento da primeira parcela
parcelas_main = parcelas_main.withColumn(
    "data_vencimento_primeira_parcela",
    F.first("data_vencimento", ignorenulls=True).over(w_primeira)
)


In [27]:
parcelas_main = parcelas_main.withColumn(
    "qtd_dias_de_atraso_v2",
    F.when(
        # 🔴 Cenário B — contrato nunca teve pagamento
        F.col("flag_contrato_sem_pagamento") == 1,
        F.greatest(
            F.datediff(
                F.col("data_vencimento"),
                F.col("data_vencimento_primeira_parcela")
            ),
            F.lit(0)
        )
    ).when(
        # 🟢 Cenário A — contrato teve pagamento
        (F.col("data_pagamento_aux").isNotNull()) &
        (F.col("data_vencimento") > F.col("data_pagamento_aux")),
        F.abs(
            F.datediff(
                F.col("data_vencimento"),
                F.col("data_pagamento_aux")
            )
        )
    ).otherwise(F.lit(0))
)

#Flag inadimplência

In [28]:

parcelas_main = parcelas_main.withColumn(
    "flag_inadimplencia_30_days",
    when(col("qtd_dias_de_atraso_v2")>30,1).otherwise(0)).withColumn(
     "flag_inadimplencia_90_days",when(col("qtd_dias_de_atraso_v2")>90,1).otherwise(0))

#parcelas_main.show(5, truncate=False)

In [29]:
#Validando flag-inadimplencia over todas parcelas
parcelas_main.select(
    "id_contrato",
    "parcela",
    "data_vencimento",
    "data_pagamento",
    "data_pagamento_aux",
    "flag_contrato_sem_pagamento",
    "qtd_dias_de_atraso_v2",
    "flag_inadimplencia_30_days",
    "flag_inadimplencia_90_days"
).orderBy("data_vencimento").filter(F.col("id_contrato") == "{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}").show(100, truncate=False)
#{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}
#{00012C6D-DD36-401A-8B26-F0C46C121114}

+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+---------------------+--------------------------+--------------------------+
|id_contrato                           |parcela|data_vencimento|data_pagamento|data_pagamento_aux|flag_contrato_sem_pagamento|qtd_dias_de_atraso_v2|flag_inadimplencia_30_days|flag_inadimplencia_90_days|
+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+---------------------+--------------------------+--------------------------+
|{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}|1      |2024-11-27     |2024-11-17    |2024-11-17        |0                          |10                   |0                         |0                         |
|{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}|2      |2024-12-27     |NULL          |2024-11-17        |0                          |40                   |1                         |0            

In [30]:
#parcelas_main.orderBy("data_vencimento").filter(F.col("id_contrato") == "{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}").show(100, truncate=False)

# Extrair Mes e Ano do Vencimento da Parcela

In [31]:
parcelas_main = parcelas_main.withColumn('year',F.year(F.to_timestamp('data_vencimento', 'yyyy-MM-dd')))
parcelas_main = parcelas_main.withColumn('month',F.month(F.to_timestamp('data_vencimento', 'yyyy-MM-dd')))

## PCT - Juros e Amortização

In [32]:
spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set("spark.sql.legacy.allowDivideByZero", "true")

#Preenchendo nulos com zero
parcelas_main = parcelas_main.fillna(0)

parcelas_main = parcelas_main.withColumn(
    "pct_juros",
    F.when(
        (F.col("valor_parcela").isNotNull()) & (F.col("valor_juros") !=0) & (F.col("valor_parcela") != 0),
        F.round(F.col("valor_juros") / F.col("valor_parcela"), 2)
    ).otherwise(F.lit(0))
)

parcelas_main = parcelas_main.withColumn(
    "pct_amortizacao",
    F.when(
        (F.col("valor_parcela").isNotNull()) & (F.col("valor_amortizacao") !=0) & (F.col("valor_parcela") != 0),
        F.round(F.col("valor_amortizacao") / F.col("valor_parcela"), 2)
    ).otherwise(F.lit(0))
)

In [33]:
parcelas_main.show(10, truncate=False)

+--------------------------------------+--------------+---------------+-------+---------+------------------------------+------------+-------------+------------+-----------+-----------+-----------------+----------+--------------+------------------+--------------------------+----------------+--------------------------------------+-------+------------------+-------------------------------+--------------------------------------+--------------------------------------+---------------+----------------------------------------------------------------+----------------+------------------+---------------------------+--------------------------------+---------------------+--------------------------+--------------------------+----+-----+---------+---------------+
|id_contrato                           |data_pagamento|data_vencimento|valor  |valor_iof|valor_financiado_principal_iof|qtd_parcelas|valor_parcela|valor_tarifa|valor_juros|valor_iof_2|valor_amortizacao|valor_pago|valor_desconto|valor_juros_a

# Feature Selection

================== # VARIAVEIS DE ALAVANCAGEM # ========================

flag_contrato_sem_pagamento (Avaliar se o não pagamento da primeira parcela acarreta em inadimplência)

qtd_parcelas

valor

valor_parcela

valor_financiado_principal_iof

valor_iof

valor_tarifa

=====================================================================

================== # VARIAVEIS DE SAZONALIDADE # ========================

parcela_norm_0_1

mes_vencimento * transformar a variael

ano_vencimento * transformar a variavel

=====================================================================

================== # VARIAVEIS DE DECOMPOSIÇÃO FINANCEIRA # ========================

valor_amortizacao

valor_juros

valor_juros_remuneratorios

valor_iof_2

E razões (features derivadas):

* pct_juros = valor_juros / valor_parcela

* pct_amortizacao = valor_amortizacao / valor_parcela

Justificativa: Parcelas mais “carregadas de juros” tendem a inadimplir mais.

=====================================================================


================== # VARIAVEIS DE DECOMPOSIÇÃO FINANCEIRA # =============



In [ ]:
#https://archive.is/03ptu


# Transformar em Pandas Dataframe

In [43]:
parcelas_main_pd = parcelas_main.toPandas()

parcelas_main_pd_aux = parcelas_main_pd[["qtd_parcelas",
    "valor",
    "valor_parcela",
    "valor_financiado_principal_iof",
    "valor_iof",
    "valor_tarifa",
    "year",
    "month",
    "parcela_norm_0_1",
    "valor_amortizacao",
    "valor_juros",
    "valor_juros_remuneratorios",
    "valor_iof_2",
    "pct_juros",
    "pct_amortizacao",
    "flag_inadimplencia_90_days",
    ]].copy()

parcelas_main_pd_aux = parcelas_main_pd_aux.dropna()


X = parcelas_main_pd_aux[["qtd_parcelas",
    "valor",
    "valor_parcela",
    "valor_financiado_principal_iof",
    "valor_iof",
    "valor_tarifa",
    "year",
    "month",
    "parcela_norm_0_1",
    "valor_amortizacao",
    "valor_juros",
    "valor_juros_remuneratorios",
    "pct_juros",
    "pct_amortizacao",
    "valor_iof_2"
    ]].copy()
y = parcelas_main_pd_aux["flag_inadimplencia_90_days"]

# Foward Selection

In [37]:
def forward_selection_simple(X, y, threshold=0.05):
  selected_features = []
  feature_pvalues = {}
  #X = X.reset_index(drop=True)
  #y = y.reset_index(drop=True)
  while True:
    remaining_features = [f for f in X.columns if f not in selected_features]
    new_pval = pd.Series(index=remaining_features)
    for f in remaining_features:
      model = sm.OLS(y, sm.add_constant(X[selected_features + [f]])).fit() # + f ou + features
      new_pval[f] = model.pvalues[f]
    min_pval = new_pval.min()
    if min_pval < 0.05:
      selected_features.append(new_pval.idxmin())
    else:
      break
  return selected_features

selected = forward_selection_simple(X,y)
print(f"Selected features: {selected}")

Selected features: ['qtd_parcelas', 'valor_tarifa', 'valor', 'year', 'month', 'parcela_norm_0_1', 'valor_amortizacao', 'valor_parcela', 'valor_financiado_principal_iof', 'valor_iof', 'valor_juros', 'pct_juros', 'pct_amortizacao', 'valor_iof_2', 'valor_juros_remuneratorios']


In [38]:
def forward_selection_simple(X, y, threshold=0.05):
    selected_features = []
    feature_pvalues = {}

    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    while True:
        remaining_features = [f for f in X.columns if f not in selected_features]
        if not remaining_features:
            break

        new_pval = pd.Series(index=remaining_features, dtype=float)

        for f in remaining_features:
            X_temp = X[selected_features + [f]] if selected_features else X[[f]]
            model = sm.OLS(y, sm.add_constant(X_temp)).fit()
            new_pval[f] = model.pvalues[f]

        min_pval = new_pval.min()

        if min_pval < threshold:
            best_feature = new_pval.idxmin()
            selected_features.append(best_feature)
            feature_pvalues[best_feature] = min_pval
            print(f"✓ {best_feature}: p-value = {min_pval:.6f}")
        else:
            break

    return selected_features, feature_pvalues

selected, pvalues = forward_selection_simple(X, y)

✓ qtd_parcelas: p-value = 0.000000
✓ valor_tarifa: p-value = 0.000000
✓ valor: p-value = 0.000000
✓ year: p-value = 0.000000
✓ month: p-value = 0.000000
✓ parcela_norm_0_1: p-value = 0.000000
✓ valor_amortizacao: p-value = 0.000000
✓ valor_parcela: p-value = 0.000000
✓ valor_financiado_principal_iof: p-value = 0.000000
✓ valor_iof: p-value = 0.000000
✓ valor_juros: p-value = 0.000000
✓ pct_juros: p-value = 0.000000
✓ pct_amortizacao: p-value = 0.000000
✓ valor_iof_2: p-value = 0.000000
✓ valor_juros_remuneratorios: p-value = 0.000000


# Backward Selection

In [62]:
def backward_selection_simple(X,y, threshold=0.05):
  selected_features = list(X.columns)
  feature_pvalues = {}

  X = X.reset_index(drop=True)
  y = y.reset_index(drop=True)

  while True:
    X_temp = sm.add_constant(X[selected_features])
    model = sm.OLS(y, X_temp).fit()

    #remoção do intercepto
    pvalues = model.pvalues.drop("const")

    #seleciona o p-value maior restante
    max_pval = pvalues.max()

    if max_pval > threshold:
      worst_feature = pvalues.idxmax()
      feature_pvalues.append({
          "remove_feature": worst_feature,
          "pvalue": max_pval
      })

      feature_pvalues[worst_feature] = max_pval
      selected_features.remove(worst_feature)

      print(f"✗ {worst_feature}: p-value = {max_pval:.6f}")
    else:
      break
  X_final = sm.add_constant(X[selected_features])
  final_model = sm.OLS(y, X_final).fit()

  summary_df = pd.DataFrame({
      "feature": final_model.params.index,
      "coefficient": final_model.params.values,
      "std_error": final_model.bse.values,

      "pvalue": final_model.pvalues.values,
      "tvalue": final_model.tvalues.values,
      "rsquared": final_model.rsquared,
      "coef": final_model.params.values
  })
  # Intervalo de confiança 95%
  conf_int = final_model.conf_int()
  summary_df["ci_lower_95"] = conf_int[0].values
  summary_df["ci_upper_95"] = conf_int[1].values

  # remove intercepto da tabela final
  summary_df = summary_df[summary_df["feature"] != "const"].reset_index(drop=True)

  # -------------------------
  # Métricas globais
  # -------------------------
  model_metrics = pd.DataFrame({
  "metric": ["R2_adj", "AIC", "BIC"],
  "value": [
      final_model.rsquared_adj,
      final_model.aic,
      final_model.bic
  ]
  })

  # histórico de remoções
  removed_df = pd.DataFrame(feature_pvalues)

  return selected_features, removed_df, summary_df, model_metrics


In [64]:
selected_features, removed_df, summary_df, model_metrics = backward_selection_simple(X,y,threshold=0.05)


In [65]:
summary_df

,feature,coefficient,std_error,pvalue,tvalue,rsquared,coef,ci_lower_95,ci_upper_95
0,qtd_parcelas,0.021320,0.000102,0.000000e+00,209.963239,0.213465,0.021320,0.021121,0.021519
1,valor,0.000761,0.000032,5.700218e-123,23.586689,0.213465,0.000761,0.000698,0.000824
2,valor_parcela,0.000937,0.000031,5.526185e-203,30.406754,0.213465,0.000937,0.000877,0.000997
3,valor_financiado_principal_iof,-0.000873,0.000032,3.722944e-166,-27.477821,0.213465,-0.000873,-0.000935,-0.000810
4,valor_iof,-0.001634,0.000064,4.935389e-144,-25.558297,0.213465,-0.001634,-0.001759,-0.001508
5,valor_tarifa,-0.021157,0.000195,0.000000e+00,-108.238558,0.213465,-0.021157,-0.021540,-0.020774
6,year,0.017489,0.001306,6.406349e-41,13.396304,0.213465,0.017489,0.014931,0.020048
7,month,0.001598,0.000103,1.440986e-54,15.557278,0.213465,0.001598,0.001397,0.001799
8,parcela_norm_0_1,0.068813,0.000813,0.000000e+00,84.650435,0.213465,0.068813,0.067220,0.070406
9,valor_amortizacao,-0.001919,0.000035,0.000000e+00,-54.251102,0.213465,-0.001919,-0.001988,-0.001850


# VIF + Backward Selection

In [44]:
def backward_selection_VIF(X, y, threshold=0.05):

    selected_features = list(X.columns)
    removed_features = []

    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    # =========================
    # BACKWARD SELECTION
    # =========================
    while True:
        X_temp = sm.add_constant(X[selected_features])
        model = sm.OLS(y, X_temp).fit()

        pvalues = model.pvalues.drop("const")
        max_pval = pvalues.max()

        if max_pval > threshold:
            worst_feature = pvalues.idxmax()
            removed_features.append({
                "removed_feature": worst_feature,
                "pvalue": max_pval
            })
            selected_features.remove(worst_feature)
            print(f"✗ {worst_feature}: p-value = {max_pval:.6f}")
        else:
            break

    # =========================
    # MODELO FINAL
    # =========================
    X_final = sm.add_constant(X[selected_features])
    final_model = sm.OLS(y, X_final).fit()

    # =========================
    # SUMMARY POR VARIÁVEL
    # =========================
    summary_df = pd.DataFrame({
        "feature": final_model.params.index,
        "coef": final_model.params.values,
        "std_error": final_model.bse.values,
        "tvalue": final_model.tvalues.values,
        "pvalue": final_model.pvalues.values
    })

    conf_int = final_model.conf_int()
    summary_df["ci_lower_95"] = conf_int[0].values
    summary_df["ci_upper_95"] = conf_int[1].values

    summary_df = summary_df[summary_df["feature"] != "const"].reset_index(drop=True)

    # =========================
    # VIF (SEM INTERCEPTO)
    # =========================
    vif_data = pd.DataFrame({
        "feature": selected_features,
        "VIF": [
            variance_inflation_factor(X[selected_features].values, i)
            for i in range(len(selected_features))
        ]
    })

    summary_vif = summary_df.merge(vif_data, on="feature", how="inner")

    # =========================
    # MÉTRICAS GLOBAIS
    # =========================
    model_metrics = pd.DataFrame({
        "metric": ["R2_adj", "AIC", "BIC"],
        "value": [
            final_model.rsquared_adj,
            final_model.aic,
            final_model.bic
        ]
    })

    removed_df = pd.DataFrame(removed_features)

    return selected_features, removed_df, summary_df, model_metrics, summary_vif


In [45]:
selected_features, removed_df, summary_df, model_metrics,summary_vf = backward_selection_VIF(X, y, threshold=0.05)
#summary_vf

/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


In [48]:
summary_vf["VIF"] = summary_vf["VIF"].replace([np.inf, -np.inf],0)
summary_vf = summary_vf[summary_vf["VIF"] < 10]


In [42]:
X.columns

Index(['qtd_parcelas', 'valor', 'valor_parcela',
       'valor_financiado_principal_iof', 'valor_iof', 'valor_tarifa', 'year',
       'month', 'parcela_norm_0_1', 'parcela_norm_0_1', 'valor_amortizacao',
       'valor_juros', 'valor_juros_remuneratorios', 'pct_juros',
       'pct_amortizacao', 'valor_iof_2', 'parcela_norm_0_1',
       'parcela_norm_0_1'],
      dtype='object')

In [53]:
summary_vf

,feature,coef,std_error,tvalue,pvalue,ci_lower_95,ci_upper_95,VIF
1,valor,0.001913,0.000029,66.379927,0.000000e+00,0.001857,0.001970,0.000000
3,valor_financiado_principal_iof,-0.002087,0.000029,-72.963061,0.000000e+00,-0.002143,-0.002031,0.000000
4,valor_iof,-0.004000,0.000057,-69.742190,0.000000e+00,-0.004112,-0.003887,0.000000
5,valor_tarifa,-0.019833,0.000181,-109.807523,0.000000e+00,-0.020187,-0.019479,1.616600
7,month,0.003465,0.000089,38.853650,0.000000e+00,0.003290,0.003640,6.328164
8,parcela_norm_0_1,-0.076566,0.000897,-85.338259,0.000000e+00,-0.078324,-0.074807,5.505104
11,valor_juros_remuneratorios,-0.000822,0.000077,-10.739349,6.672361e-27,-0.000973,-0.000672,1.418951


In [56]:
pd.set_option('display.max_columns', None) # Mostra todas as colunas
#parcelas_main_pd.head(5)

#Dimensões
#id_contrato
#data_pagamento
#data_vencimento
#id_parcela
#uuid_cliente
#id_contrato_original
#id_contrato_pai
#cpf_hash_sha256

In [62]:
feature_selected = summary_vf["feature"].tolist()

cols_id = [
    "id_contrato",
    "data_pagamento",
    "data_vencimento",
    "id_parcela",
    "uuid_cliente",
    "id_contrato_original",
    "id_contrato_pai",
    "cpf_hash_sha256",
    "flag_inadimplencia_90_days",
    "flag_inadimplencia_30_days"
]

parcelas_main_pd_filtrado = parcelas_main_pd[cols_id + feature_selected].copy()


In [63]:
parcelas_main_pd_filtrado.head(3)

,id_contrato,data_pagamento,data_vencimento,id_parcela,uuid_cliente,id_contrato_original,id_contrato_pai,cpf_hash_sha256,flag_inadimplencia_90_days,flag_inadimplencia_30_days,valor,valor_financiado_principal_iof,valor_iof,valor_tarifa,month,parcela_norm_0_1,valor_juros_remuneratorios
0,{00024A4D-0B45-49CA-8289-44642C4DDD7E},2025-09-09,2025-09-12,{3648CD2A-1BC0-4009-A8D7-1AC396DBBA8C},{AA8760DA-BB29-4589-840C-554C4C92CA0E},{00024A4D-0B45-49CA-8289-44642C4DDD7E},None,62f0b299d8a2993b817f68b6bac50285fb7927100d4e41...,0,0,681.65,687.77,6.12,0.0,9,0.0,0.0
1,{00024A4D-0B45-49CA-8289-44642C4DDD7E},2025-10-11,2025-10-12,{C94698B4-B38F-4C96-AAB9-9A88BEB6D32D},{AA8760DA-BB29-4589-840C-554C4C92CA0E},{00024A4D-0B45-49CA-8289-44642C4DDD7E},None,62f0b299d8a2993b817f68b6bac50285fb7927100d4e41...,0,0,681.65,687.77,6.12,0.0,10,1.0,0.0
2,{0002AD5E-2A67-47D7-B314-C863E1ED9BCB},2025-08-15,2025-08-14,{C7A5DCC2-0F24-4E78-836F-E7A8A0B2EC47},{6A7B8032-5CE3-4A6D-8DC5-E95B1DBD7630},{0002AD5E-2A67-47D7-B314-C863E1ED9BCB},None,f23157a283e15fbc8b6842401e0e25e830f22d1921764e...,0,0,3655.83,3693.74,37.91,0.0,8,0.0,0.0


In [64]:
len(parcelas_main_pd_filtrado)

1063816

In [65]:
base_path = "/content/drive/MyDrive/MBA - Ciencia de Dados - USP/dados_tcc/"

file_name = "parcelas_main_pd_filtrado.csv"

parcelas_main_pd_filtrado.to_csv(
    base_path + file_name,
    index=False,
    encoding="utf-8"
)
